In [58]:
import numpy as np
from scipy.interpolate import interp1d,RegularGridInterpolator


In [77]:
def get_fraction_for_all_binaries(rho_data, time):
    """
        rho_data: stellar density of star cluster in half-mass radius [M_sun / pc^3]
        time: evolutionary time [Myr]
    """
    def get_k1(rho_data):
        data = np.loadtxt("all_binary.txt")
        rho,k1 = data[0],data[1]
        min_rho, max_rho = rho.min(),rho.max()
        if rho_data < min_rho:
            print("rho smaller than the range of our data, your rho will be regularized.")
            rho_data = min_rho
        if rho_data > max_rho:
            print("rho bigger than the range of our data, your rho will be regularized.")
            rho_data = max_rho
        log_rho,log_k1 = np.log(rho),np.log(np.abs(k1))
        k1_interp = interp1d(log_rho, log_k1,kind='linear') 
        k1_new = -np.exp(k1_interp(np.log(rho_data)))
        return k1_new
    def get_tb(rho_data):
        data = np.loadtxt("all_binary_tb.txt")
        rho,tb = data[0],data[1]
        min_rho, max_rho = rho.min(),rho.max()
        if rho_data < min_rho:
       
            rho_data = min_rho
        if rho_data > max_rho:
           
            rho_data = max_rho
        log_rho,log_tb = np.log(rho),np.log(tb)
        tb_interp = interp1d(log_rho, log_tb,kind='linear') 
        tb_new = np.exp(tb_interp(np.log(rho_data)))
        return tb_new
    k1 = get_k1(rho_data)
    tb = get_tb(rho_data)
    if time > tb:
        time = tb
        # print("time > tb")
    fraction = k1 * time + 1
    return fraction

In [78]:
get_fraction_for_all_binaries(70,2) # rho=70 M_sun/pc^3 day time=2Myr

0.9585000615110593

In [79]:
def get_fraction_for_binaries_with_period(rho_data,p_data,time):    
    """
        rho_data: stellar density of star cluster in half-mass radius [M_sun / pc^3]
        p_data: period of binaries population [log_{10} (P/day)]
        time: evolutionary time [Myr]
    """
    bins=21
    def get_k1(rho_data,p_data): 
        data = np.load("p_k1.npy")
        all_rho = data[0]
        all_k1 = data[1]
        min_rho, max_rho = all_rho.min(),all_rho.max()
        if rho_data < min_rho:
            print("rho smaller than the range of our data, your rho will be regularized.")
            rho_data = min_rho
        if rho_data > max_rho:
            print("rho bigger than the range of our data, your rho will be regularized.")
            rho_data = max_rho
        
        
        p_range = [
                (5.0,7.0),(5.1,7.1),(5.2,7.2),(5.3,7.3),(5.4,7.4),(5.5,7.5),(5.6,7.6),
                (5.7,7.7),(5.8,7.8),(5.9,7.9),(6.0,8.0),(6.1,8.1),(6.2,8.2),(6.3,8.3),
                (6.4,8.4),(6.5,8.5),(6.6,8.6),(6.7,8.7),(6.8,8.8),(6.9,8.9),(7.0,9.0)
                    ]
        
        p_vals = np.median(np.array(p_range), axis=1)
        if p_data < p_vals.min():
            print("period smaller than the range of our data, your period will be regularized.")
            p_data = p_vals.min()
        if p_data > p_vals.max():
            print("period bigger than the range of our data, your period will be regularized.")
            p_data = p_vals.max()
        # 根据rho，拿出21个模型的k1值以及对应的p，然后做一个插值模型当前rho下的p,k1插值模型，再根据给定的p进行插值，得到k1
        all_inp_k1 = []
        for i in range(bins):
            rho = all_rho[i]
            k1 = all_k1[i]
            log_rho,log_k1 = np.log(rho),np.log(np.abs(k1))

            # 做当前的rho,k1插值模型
            cur_rho_k1_inp = interp1d(log_rho,log_k1,'linear')
            # 把当前的rho放进去，得到当前p下的k1
            cur_k1 = -np.exp(cur_rho_k1_inp(np.log(rho_data)))
            all_inp_k1.append(cur_k1)
        all_inp_k1=np.array(all_inp_k1)
        # p_vals和all_k1就是用来做插值模型的
        p_k1_inp = interp1d(p_vals,all_inp_k1,'linear')

        k1 = p_k1_inp(p_data)
        return k1

    def get_tb(rho_data,p_data):
        data = np.load("p_tb.npy")
        all_rho = data[0]
        all_tb = data[1]
        min_rho, max_rho = all_rho.min(),all_rho.max()
        if rho_data < min_rho:
       
            rho_data = min_rho
        if rho_data > max_rho:
           
            rho_data = max_rho
        p_range = [
                (5.0,7.0),(5.1,7.1),(5.2,7.2),(5.3,7.3),(5.4,7.4),(5.5,7.5),(5.6,7.6),
                (5.7,7.7),(5.8,7.8),(5.9,7.9),(6.0,8.0),(6.1,8.1),(6.2,8.2),(6.3,8.3),
                (6.4,8.4),(6.5,8.5),(6.6,8.6),(6.7,8.7),(6.8,8.8),(6.9,8.9),(7.0,9.0)
                    ]
        p_vals = np.median(np.array(p_range), axis=1)
        if p_data < p_vals.min():
            p_data = p_vals.min()
        if p_data > p_vals.max():
            p_data = p_vals.max()
        all_inp_tb = []
        for i in range(bins):
            rho = all_rho[i]
            tb = all_tb[i]
            log_rho,log_tb = np.log(rho),np.log(tb)
            cur_rho_tb_inp = interp1d(log_rho, log_tb,kind='linear') 
            cur_tb = np.exp(cur_rho_tb_inp(np.log(rho_data)))
            all_inp_tb.append(cur_tb)
        all_inp_tb = np.array(all_inp_tb)
        p_tb_inp = interp1d(p_vals, all_inp_tb,'linear')

        tb = p_tb_inp(p_data)
        return tb
    k1 = get_k1(rho_data,p_data)
    tb = get_tb(rho_data,p_data)
    if time > tb:
        time = tb
    fraction = k1 * time +1
    return fraction

In [80]:
get_fraction_for_binaries_with_period(70, 8, 3) # rho=70 M_sun/pc^3 period=10**8 day time=3Myr

0.8022946100001749

In [81]:
def get_fraction_for_binaries_with_mass_ratio(rho_data,q_data,time):    
    """
        rho_data: stellar density of star cluster in half-mass radius [M_sun / pc^3]
        q_data: mass ratio of binaries population
        time: evolutionary time [Myr]
    """
    bins=17
    def get_k1(rho_data,q_data): 
        data = np.load("q_k1.npy")
        all_rho = data[0]
        all_k1 = data[1]
        bins=17
        min_rho, max_rho = all_rho.min(),all_rho.max()
        if rho_data < min_rho:
            print("rho smaller than the range of our data, your rho will be regularized.")
            rho_data = min_rho
        if rho_data > max_rho:
            print("rho bigger than the range of our data, your rho will be regularized.")
            rho_data = max_rho
        q_range = [
                (0.00,0.20),(0.05,0.25),(0.10,0.30),(0.15,0.35),(0.20,0.40),
                (0.25,0.45),(0.30,0.50),(0.35,0.55),(0.40,0.60),(0.45,0.65),
                (0.50,0.70),(0.55,0.75),(0.60,0.80),(0.65,0.85),(0.70,0.90),
                (0.75,0.95),(0.8,1.0)]
        q_vals = np.median(np.array(q_range),axis=1)
        if q_data < q_vals.min():
            print("mass ratio smaller than the range of our data, your mass ratio will be regularized.")
            q_data = q_vals.min()
        if q_data > q_vals.max():
            print("mass ratio bigger than the range of our data, your mass ratio will be regularized.")
            q_data = q_vals.max()
        all_inp_k1 = []
        for i in range(bins):
            rho = all_rho[i]
            k1 = all_k1[i]
            log_rho,log_k1 = np.log(rho),np.log(np.abs(k1))
            cur_rho_k1_inp = interp1d(log_rho,log_k1,'linear')
            cur_k1 = -np.exp(cur_rho_k1_inp(np.log(rho_data)))
            all_inp_k1.append(cur_k1)
        all_inp_k1=np.array(all_inp_k1)
        q_k1_inp = interp1d(q_vals,all_inp_k1,'linear')
        k1 = q_k1_inp(q_data)
        return k1
    def get_tb(rho_data,q_data):
        data = np.load("q_tb.npy")
        all_rho = data[0]
        all_tb = data[1]
        bins=17
        min_rho, max_rho = all_rho.min(),all_rho.max()
        if rho_data < min_rho:
            rho_data = min_rho
        if rho_data > max_rho:
            rho_data = max_rho
        q_range = [
                (0.00,0.20),(0.05,0.25),(0.10,0.30),(0.15,0.35),(0.20,0.40),
                (0.25,0.45),(0.30,0.50),(0.35,0.55),(0.40,0.60),(0.45,0.65),
                (0.50,0.70),(0.55,0.75),(0.60,0.80),(0.65,0.85),(0.70,0.90),
                (0.75,0.95),(0.8,1.0)]
        q_vals = np.median(np.array(q_range),axis=1)
        if q_data < q_vals.min():
            q_data = q_vals.min()
        if q_data > q_vals.max():
            q_data = q_vals.max()
        all_inp_tb = []
        for i in range(bins):
            rho = all_rho[i]
            tb = all_tb[i]
            log_rho,log_tb = np.log(rho),np.log(tb)
            cur_rho_tb_inp = interp1d(log_rho, log_tb,kind='linear') 
            cur_tb = np.exp(cur_rho_tb_inp(np.log(rho_data)))
            all_inp_tb.append(cur_tb)
        all_inp_tb = np.array(all_inp_tb)
        p_tb_inp = interp1d(q_vals, all_inp_tb,'linear')

        tb = p_tb_inp(q_data)
        return tb
    k1 = get_k1(rho_data,q_data)
    tb = get_tb(rho_data,q_data)
    if time > tb:
        time = tb
    fraction = k1 * time +1
    return fraction

In [82]:
get_fraction_for_binaries_with_mass_ratio(2000,0.6,3) # rho=2000 M_sun/pc^3 mass ratio=0.6 time=3Myr

0.7335406943354938

In [83]:
def get_fraction_for_binaries_with_p_q(rho_data,p_data,q_data,time):
    """
        rho_data: stellar density of star cluster in half-mass radius [M_sun / pc^3]
        p_data: period of binaries population [log_{10} (P/day)]
        q_data: mass ratio of binaries population
        time: evolutionary time [Myr]
    """
    bins=231
    def get_k1(rho_data,p_data,q_data):
        period_bins = np.array([
            (5.0,7.0),(5.1,7.1),(5.2,7.2),(5.3,7.3),(5.4,7.4),(5.5,7.5),(5.6,7.6),
            (5.7,7.7),(5.8,7.8),(5.9,7.9),(6.0,8.0),(6.1,8.1),(6.2,8.2),(6.3,8.3),
            (6.4,8.4),(6.5,8.5),(6.6,8.6),(6.7,8.7),(6.8,8.8),(6.9,8.9),(7.0,9.0)
                ])
        q_bins = np.array([
            (0.0,0.50),(0.05,0.55),(0.10,0.60),(0.15,0.65),(0.20,0.70),(0.25,0.75),
            (0.30,0.80),(0.35,0.85),(0.40,0.90),(0.45,0.95),(0.50,1.0)
        ])
        p_vals = np.median(np.array(period_bins),axis=1)
        q_vals = np.median(np.array(q_bins),axis=1)


        data = np.load("p_q_k1.npy")
        all_rho = data[0]
        all_k1 = data[1]

        min_rho, max_rho = all_rho.min(),all_rho.max()
        if rho_data < min_rho:
            print("rho smaller than the range of our data, your rho will be regularized.")
            rho_data = min_rho
        if rho_data > max_rho:
            print("rho bigger than the range of our data, your rho will be regularized.")
            rho_data = max_rho
        if p_data < p_vals.min():
            print("period smaller than the range of our data, your period will be regularized.")
            p_data = p_vals.min()
        if p_data > p_vals.max():
            print("period bigger than the range of our data, your period will be regularized.")
            p_data = p_vals.max()
        if q_data < q_vals.min():
            print("mass ratio smaller than the range of our data, your mass ratio will be regularized.")
            q_data = q_vals.min()
        if q_data > q_vals.max():
            print("mass ratio bigger than the range of our data, your mass ratio will be regularized.")
            q_data = q_vals.max()
        all_inp_k1 = []
        for i in range(bins): # 取出所有pq区间下，处于当前rho对应的k1值
            rho = all_rho[i]
            k1 = all_k1[i]
            log_rho,log_k1 = np.log(rho),np.log(np.abs(k1))
            cur_rho_k1_inp = interp1d(log_rho,log_k1,'linear')
            cur_k1 = -np.exp(cur_rho_k1_inp(np.log(rho_data))) # 就是这个，231个bin共有231个特定rho下的k1值
            all_inp_k1.append(cur_k1)
        all_inp_k1=np.array(all_inp_k1)
        all_inp_k1 = all_inp_k1.reshape(len(p_vals),len(q_vals))


        p_q_inp = RegularGridInterpolator((p_vals,q_vals),all_inp_k1,'linear')
        # interp_func = RegularGridInterpolator((p, q), k)
        k1 = p_q_inp((p_data,q_data))
        return k1

    def get_tb(rho_data,p_data,q_data):
        period_bins = np.array([
            (5.0,7.0),(5.1,7.1),(5.2,7.2),(5.3,7.3),(5.4,7.4),(5.5,7.5),(5.6,7.6),
            (5.7,7.7),(5.8,7.8),(5.9,7.9),(6.0,8.0),(6.1,8.1),(6.2,8.2),(6.3,8.3),
            (6.4,8.4),(6.5,8.5),(6.6,8.6),(6.7,8.7),(6.8,8.8),(6.9,8.9),(7.0,9.0)
                ])
        q_bins = np.array([
            (0.0,0.50),(0.05,0.55),(0.10,0.60),(0.15,0.65),(0.20,0.70),(0.25,0.75),
            (0.30,0.80),(0.35,0.85),(0.40,0.90),(0.45,0.95),(0.50,1.0)
        ])
        p_vals = np.median(np.array(period_bins),axis=1)
        q_vals = np.median(np.array(q_bins),axis=1)

        data = np.load("p_q_tb.npy")
        all_rho = data[0]
        all_tb = data[1]

        all_inp_tb = []
        min_rho, max_rho = all_rho.min(),all_rho.max()
        if rho_data < min_rho:
            rho_data = min_rho
        if rho_data > max_rho:
            rho_data = max_rho
        if p_data < p_vals.min():
            p_data = p_vals.min()
        if p_data > p_vals.max():
            p_data = p_vals.max()
        if q_data < q_vals.min():
            q_data = q_vals.min()
        if q_data > q_vals.max():
            q_data = q_vals.max()
        for i in range(bins): 
            rho = all_rho[i]
            tb = all_tb[i]
            log_rho,log_tb = np.log(rho),np.log(np.abs(tb))
            cur_rho_tb_inp = interp1d(log_rho,log_tb,'linear')
            cur_tb = np.exp(cur_rho_tb_inp(np.log(rho_data)))
            all_inp_tb.append(cur_tb)
        all_inp_tb=np.array(all_inp_tb)
        all_inp_tb = all_inp_tb.reshape(len(p_vals),len(q_vals))


        p_q_inp = RegularGridInterpolator((p_vals,q_vals),all_inp_tb,'linear')
        tb = p_q_inp((p_data,q_data))
        return tb
    k1 = get_k1(rho_data,p_data,q_data)
    tb = get_tb(rho_data,p_data,q_data)
    if time > tb:
        time = tb
    fraction = k1 * time +1
    return fraction

In [84]:
get_fraction_for_binaries_with_p_q(1800,6.5,0.75,3) # rho=1800 M_sun/pc^3 period=10**6.5 day mass ratio=0.75 time=3Myr

0.6247746405207146